In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Get project root
cwd = Path.cwd()
root = cwd.parent if cwd.name == "src" else cwd

# Load datasets prepared by Member 1
train = pd.read_csv(root / "data" / "processed" / "train.csv")
valid = pd.read_csv(root / "data" / "processed" / "valid.csv")
test = pd.read_csv(root / "data" / "processed" / "test.csv")

print("Train shape:", train.shape)
print("Validation shape:", valid.shape)
print("Test shape:", test.shape)

train.head()

Train shape: (484, 15)
Validation shape: (123, 15)
Test shape: (122, 15)


,season,yr,mnth,holiday,weekday,workingday,weathersit_lag2,temp_lag2,atemp_lag2,hum_lag2,windspeed_lag2,casual_lag2,registered_lag2,cnt_lag2,cnt
0,1,0,1,0,1,1,2.0,0.344167,0.363625,0.805833,0.160446,331.0,654.0,985.0,1349
1,1,0,1,0,2,1,2.0,0.363478,0.353739,0.696087,0.248539,131.0,670.0,801.0,1562
2,1,0,1,0,3,1,1.0,0.196364,0.189405,0.437273,0.248309,120.0,1229.0,1349.0,1600
3,1,0,1,0,4,1,1.0,0.200000,0.212122,0.590435,0.160296,108.0,1454.0,1562.0,1606
4,1,0,1,0,5,1,1.0,0.226957,0.229270,0.436957,0.186900,82.0,1518.0,1600.0,1510


In [2]:
print(train.columns.tolist())

['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit_lag2', 'temp_lag2', 'atemp_lag2', 'hum_lag2', 'windspeed_lag2', 'casual_lag2', 'registered_lag2', 'cnt_lag2', 'cnt']


In [3]:
target = "cnt"

features = [
    "season",
    "yr",
    "mnth",
    "holiday",
    "weekday",
    "workingday",
    "weathersit_lag2",
    "temp_lag2",
    "atemp_lag2",
    "hum_lag2",
    "windspeed_lag2",
    "casual_lag2",
    "registered_lag2",
    "cnt_lag2"
]

X_train = train[features]
y_train = train[target]

X_valid = valid[features]
y_valid = valid[target]

X_test = test[features]
y_test = test[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (484, 14)
y_train: (484,)
X_valid: (123, 14)
y_valid: (123,)
X_test: (122, 14)
y_test: (122,)


In [19]:
# Feature engineering: cyclical encoding for month

X_train = X_train.copy()
X_valid = X_valid.copy()
X_test = X_test.copy()

for df in [X_train, X_valid, X_test]:
    df["month_sin"] = np.sin(2 * np.pi * df["mnth"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["mnth"] / 12)

print("Feature engineering completed.")
print("X_train shape:", X_train.shape)
print("New features:", ["month_sin", "month_cos"])

Feature engineering completed.
X_train shape: (484, 16)
New features: ['month_sin', 'month_cos']


In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "season",
    "yr",
    "mnth",
    "holiday",
    "weekday",
    "workingday",
    "weathersit_lag2"
]

numeric_features = [
    "temp_lag2",
    "atemp_lag2",
    "hum_lag2",
    "windspeed_lag2",
    "casual_lag2",
    "registered_lag2",
    "cnt_lag2",
    "month_sin",
    "month_cos"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_valid)

linear_mae = mean_absolute_error(y_valid, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_valid, linear_pred))
linear_r2 = r2_score(y_valid, linear_pred)

print("Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

Linear Regression
MAE: 744.388797214908
RMSE: 1106.162771030896
R²: -0.38075406167640513


In [23]:
from sklearn.linear_model import Ridge
from sklearn.base import clone

ridge_results = []

for alpha in [0.01, 0.1, 1, 10, 100]:

    ridge_model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_model.fit(X_train, y_train)

    ridge_pred = ridge_model.predict(X_valid)

    mae = mean_absolute_error(y_valid, ridge_pred)
    rmse = np.sqrt(mean_squared_error(y_valid, ridge_pred))
    r2 = r2_score(y_valid, ridge_pred)

    ridge_results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df

,alpha,MAE,RMSE,R2
0,0.01,743.870928,1104.722444,-0.377161
1,0.10,739.576710,1093.010413,-0.348115
2,1.00,721.723932,1034.957053,-0.208712
3,10.00,714.665655,965.987831,-0.052983
4,100.00,947.792339,1130.504046,-0.442190


In [24]:
best_ridge_alpha = 10

best_ridge_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", Ridge(alpha=best_ridge_alpha))
])

best_ridge_model.fit(X_train, y_train)

best_ridge_pred = best_ridge_model.predict(X_valid)

best_ridge_mae = mean_absolute_error(y_valid, best_ridge_pred)
best_ridge_rmse = np.sqrt(mean_squared_error(y_valid, best_ridge_pred))
best_ridge_r2 = r2_score(y_valid, best_ridge_pred)

print("Best Ridge Model")
print("Alpha:", best_ridge_alpha)
print("MAE:", best_ridge_mae)
print("RMSE:", best_ridge_rmse)
print("R²:", best_ridge_r2)

Best Ridge Model
Alpha: 10
MAE: 714.6656545777255
RMSE: 965.9878306122465
R²: -0.05298349624886023


In [25]:
from sklearn.linear_model import Lasso

lasso_results = []

for alpha in [0.01, 0.1, 1, 10, 100]:

    lasso_model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Lasso(alpha=alpha, max_iter=20000))
    ])

    lasso_model.fit(X_train, y_train)

    lasso_pred = lasso_model.predict(X_valid)

    mae = mean_absolute_error(y_valid, lasso_pred)
    rmse = np.sqrt(mean_squared_error(y_valid, lasso_pred))
    r2 = r2_score(y_valid, lasso_pred)

    lasso_results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

lasso_results_df = pd.DataFrame(lasso_results)

lasso_results_df

,alpha,MAE,RMSE,R2
0,0.01,744.066243,1105.440029,-0.378950
1,0.10,741.099084,1098.682827,-0.362144
2,1.00,720.908419,1037.969332,-0.215758
3,10.00,707.633546,943.042624,-0.003554
4,100.00,1226.352407,1422.025166,-1.281880


In [26]:
best_lasso_alpha = 10

best_lasso_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", Lasso(alpha=best_lasso_alpha, max_iter=20000))
])

best_lasso_model.fit(X_train, y_train)

best_lasso_pred = best_lasso_model.predict(X_valid)

best_lasso_mae = mean_absolute_error(y_valid, best_lasso_pred)
best_lasso_rmse = np.sqrt(mean_squared_error(y_valid, best_lasso_pred))
best_lasso_r2 = r2_score(y_valid, best_lasso_pred)

print("Best Lasso Model")
print("Alpha:", best_lasso_alpha)
print("MAE:", best_lasso_mae)
print("RMSE:", best_lasso_rmse)
print("R²:", best_lasso_r2)

Best Lasso Model
Alpha: 10
MAE: 707.6335462137152
RMSE: 943.0426241647939
R²: -0.0035543533118369552


In [27]:
model_comparison = pd.DataFrame([
    {
        "Model": "Baseline",
        "MAE": 1011.20,
        "RMSE": 1336.42,
        "R2": np.nan
    },
    {
        "Model": "Linear Regression",
        "MAE": linear_mae,
        "RMSE": linear_rmse,
        "R2": linear_r2
    },
    {
        "Model": "Ridge (alpha=10)",
        "MAE": best_ridge_mae,
        "RMSE": best_ridge_rmse,
        "R2": best_ridge_r2
    },
    {
        "Model": "Lasso (alpha=10)",
        "MAE": best_lasso_mae,
        "RMSE": best_lasso_rmse,
        "R2": best_lasso_r2
    }
])

model_comparison.round(2)

,Model,MAE,RMSE,R2
0,Baseline,1011.20,1336.42,NaN
1,Linear Regression,744.39,1106.16,-0.38
2,Ridge (alpha=10),714.67,965.99,-0.05
3,Lasso (alpha=10),707.63,943.04,-0.00


In [28]:
error_analysis = valid.copy()

error_analysis["prediction"] = best_lasso_pred
error_analysis["error"] = error_analysis["cnt"] - error_analysis["prediction"]
error_analysis["abs_error"] = np.abs(error_analysis["error"])

error_analysis["error_type"] = np.where(
    error_analysis["error"] > 0,
    "Underprediction",
    "Overprediction"
)

error_summary = (
    error_analysis
    .groupby("error_type")["abs_error"]
    .agg(["count", "mean", "max"])
    .round(2)
)

error_summary

,count,mean,max
error_type,,,
Overprediction,51,871.04,3601.19
Underprediction,72,591.88,2046.98


In [29]:
workingday_error = (
    error_analysis
    .groupby("workingday")["abs_error"]
    .agg(["count", "mean", "max"])
    .round(2)
)

workingday_error

,count,mean,max
workingday,,,
0,36,814.07,2330.76
1,87,663.59,3601.19


In [30]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Bike_Sharing")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:///mlflow.db


In [31]:
import mlflow.sklearn

with mlflow.start_run(run_name="Regression_Linear"):

    # Log model information
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("validation_rows", len(X_valid))

    # Log validation metrics
    mlflow.log_metric("val_mae", linear_mae)
    mlflow.log_metric("val_rmse", linear_rmse)
    mlflow.log_metric("val_r2", linear_r2)

    # Log trained model
    mlflow.sklearn.log_model(
        linear_model,
        name="model"
    )

print("Linear Regression logged to MLflow successfully.")

Linear Regression logged to MLflow successfully.


In [32]:
with mlflow.start_run(run_name="Regression_Ridge_alpha_10"):

    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", best_ridge_alpha)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("validation_rows", len(X_valid))

    mlflow.log_metric("val_mae", best_ridge_mae)
    mlflow.log_metric("val_rmse", best_ridge_rmse)
    mlflow.log_metric("val_r2", best_ridge_r2)

    mlflow.sklearn.log_model(
        best_ridge_model,
        name="model"
    )

print("Ridge model logged to MLflow successfully.")

Ridge model logged to MLflow successfully.


In [33]:
with mlflow.start_run(run_name="Regression_Lasso_alpha_10"):

    mlflow.log_param("model_type", "Lasso")
    mlflow.log_param("alpha", best_lasso_alpha)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("validation_rows", len(X_valid))

    mlflow.log_metric("val_mae", best_lasso_mae)
    mlflow.log_metric("val_rmse", best_lasso_rmse)
    mlflow.log_metric("val_r2", best_lasso_r2)

    mlflow.sklearn.log_model(
        best_lasso_model,
        name="model"
    )

print("Lasso model logged to MLflow successfully.")

Lasso model logged to MLflow successfully.


In [34]:
# Final evaluation on the untouched test set

linear_test_pred = linear_model.predict(X_test)
ridge_test_pred = best_ridge_model.predict(X_test)
lasso_test_pred = best_lasso_model.predict(X_test)

test_results = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, linear_test_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, linear_test_pred)),
        "R2": r2_score(y_test, linear_test_pred)
    },
    {
        "Model": "Ridge (alpha=10)",
        "MAE": mean_absolute_error(y_test, ridge_test_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, ridge_test_pred)),
        "R2": r2_score(y_test, ridge_test_pred)
    },
    {
        "Model": "Lasso (alpha=10)",
        "MAE": mean_absolute_error(y_test, lasso_test_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, lasso_test_pred)),
        "R2": r2_score(y_test, lasso_test_pred)
    }
])

test_results.round(2)

,Model,MAE,RMSE,R2
0,Linear Regression,1087.88,1474.01,0.43
1,Ridge (alpha=10),1012.80,1387.80,0.50
2,Lasso (alpha=10),1016.60,1382.01,0.50


In [35]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

runs = mlflow.search_runs(
    experiment_names=["Bike_Sharing"],
    order_by=["start_time DESC"]
)

# Keep the newest run for each run name
run_map = (
    runs
    .drop_duplicates(subset=["tags.mlflow.runName"], keep="first")
    .set_index("tags.mlflow.runName")["run_id"]
    .to_dict()
)

test_metrics = {
    "Regression_Linear": {
        "test_mae": mean_absolute_error(y_test, linear_test_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, linear_test_pred)),
        "test_r2": r2_score(y_test, linear_test_pred)
    },
    "Regression_Ridge_alpha_10": {
        "test_mae": mean_absolute_error(y_test, ridge_test_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, ridge_test_pred)),
        "test_r2": r2_score(y_test, ridge_test_pred)
    },
    "Regression_Lasso_alpha_10": {
        "test_mae": mean_absolute_error(y_test, lasso_test_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, lasso_test_pred)),
        "test_r2": r2_score(y_test, lasso_test_pred)
    }
}

for run_name, metrics in test_metrics.items():
    run_id = run_map[run_name]

    for metric_name, metric_value in metrics.items():
        client.log_metric(run_id, metric_name, metric_value)

print("Test metrics added to latest regression MLflow runs successfully.")

Test metrics added to latest regression MLflow runs successfully.
